In [0]:
from pyspark.sql import functions as F

bronze_order_payments = spark.table("brazilian_ecommerce.prod.bronze_order_payments")

staging_order_payments = (
    bronze_order_payments
    .withColumn("payment_sequential", F.col("payment_sequential").cast("int"))
    .withColumn(
        "payment_type",
        F.trim(F.lower(F.regexp_replace(F.col("payment_type"), "_", " ")))
    )
    .withColumn("payment_installments", F.col("payment_installments").cast("int"))
    .withColumn("payment_value", F.col("payment_value").cast("decimal(10,2)"))
    .filter(F.col("order_id").isNotNull() & F.col("payment_sequential").isNotNull())
    .dropDuplicates(["order_id", "payment_sequential"])
    .withColumn("_loaded_at", F.current_timestamp())
)

full_table_name = "brazilian_ecommerce.prod.staging_order_payments"

(
    staging_order_payments.write
    .mode("overwrite")
    .option("comment", "order payments staging table")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

# composite primary key
spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN order_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN payment_sequential SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_staging_order_payments PRIMARY KEY (order_id, payment_sequential)
""")

# foreign key
spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT fk_staging_order_payments_order_id
    FOREIGN KEY (order_id)
    REFERENCES brazilian_ecommerce.prod.staging_orders (order_id)
""")

print(f"Loaded {staging_order_payments.count()} rows into {full_table_name}")